<a href="https://colab.research.google.com/github/Joan2022Laurente/fade/blob/main/notebooks/zImageTurboV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
#   SETUP — Dark Beast (Z-Image Turbo) + Qwen3-4B + VAE + LoRA
#   + modelos custom · descarga paralela · ControlNet Union
# ================================================================

# ── CHECKPOINTS (deja URL vacía para omitir) ───────────────────
CKPT1_URL  = "https://civitai.red/api/download/models/2957651?fileId=2837007"  # @param {type:"string"}
CKPT1_NAME = "divingZimageTurbo.safetensors"                                   # @param {type:"string"}
CKPT2_URL  = ""                                                                # @param {type:"string"}
CKPT2_NAME = "darkBeast.safetensors"                                           # @param {type:"string"}
CKPT3_URL  = ""                                                                # @param {type:"string"}
CKPT3_NAME = "ZiT.safetensors"                                                 # @param {type:"string"}
CKPT4_URL  = ""                                                                # @param {type:"string"}
CKPT4_NAME = "byStable_Yogi.safetensors"                                       # @param {type:"string"}

# ── LoRAs (deja URL vacía para omitir) ────────────────────────
LORA_HF_ENABLED = True                                                          # @param {type:"boolean"}
LORA1_URL  = ""                                                                 # @param {type:"string"}
LORA1_NAME = "lora1.safetensors"                                                # @param {type:"string"}
LORA2_URL  = "https://civitai.red/api/download/models/2486059?fileId=2374406"  # @param {type:"string"}
LORA2_NAME = "nsMASTER.safetensors"                                             # @param {type:"string"}
LORA3_URL  = ""                                                                 # @param {type:"string"}
LORA3_NAME = "lora3.safetensors"                                                # @param {type:"string"}
LORA4_URL  = "https://civitai.red/api/download/models/2486059?fileId=2374406"  # @param {type:"string"}
LORA4_NAME = "pussyOnly.safetensors"                                            # @param {type:"string"}

# ── CONTROLNET ────────────────────────────────────────────────
CONTROLNET_ENABLED     = False  # @param {type:"boolean"}
CONTROLNET_SKIP_COMPAT = True   # @param {type:"boolean"}

# ══════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil, concurrent.futures
from google.colab import userdata

CIVITAI_KEY   = userdata.get("CIVITAI_KEY")
TAILSCALE_KEY = userdata.get("TAILSCALE_KEY")
HF_TOKEN      = userdata.get("HF_TOKEN")

missing = [k for k, v in [("CIVITAI_KEY", CIVITAI_KEY), ("TAILSCALE_KEY", TAILSCALE_KEY), ("HF_TOKEN", HF_TOKEN)] if not v]
if missing: raise RuntimeError(f"❌ Faltan secrets: {', '.join(missing)}")
print("✅ Secrets cargados")

COMFY        = "/content/ComfyUI"
MODELS       = f"{COMFY}/models"
CUSTOM_NODES = f"{COMFY}/custom_nodes"
WORKFLOWS    = f"{COMFY}/user/default/workflows"
HF_Z         = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files"
HF_LORA_BASE = "https://huggingface.co/joanjlau/loraprueba0/resolve/main"

# ── Agrupamos en listas internamente
CHECKPOINTS   = [(CKPT1_URL, CKPT1_NAME), (CKPT2_URL, CKPT2_NAME),
                 (CKPT3_URL, CKPT3_NAME), (CKPT4_URL, CKPT4_NAME)]
LORAS_CIVITAI = [(LORA1_URL, LORA1_NAME), (LORA2_URL, LORA2_NAME),
                 (LORA3_URL, LORA3_NAME), (LORA4_URL, LORA4_NAME)]

# ── HELPERS ───────────────────────────────────────────────────
def run(cmd, cwd=None, check=False):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip(): print(r.stdout.strip()[-2000:])
    if check and r.returncode != 0: raise RuntimeError(f"Falló: {cmd}")
    return r.returncode

def pip_install(*pkgs):
    p = " ".join(pkgs)
    run(f'"{sys.executable}" -m pip install {p} -q --no-deps 2>/dev/null || '
        f'"{sys.executable}" -m pip install {p} -q')

def clone(url, dst):
    if os.path.isdir(dst): run("git pull --ff-only", cwd=dst)
    else:                   run(f'git clone --depth 1 "{url}" "{dst}"')
    req = f"{dst}/requirements.txt"
    if os.path.exists(req): run(f'"{sys.executable}" -m pip install -r "{req}" -q')

def file_ok(path): return os.path.exists(path) and os.path.getsize(path) > 1_048_576

def dl(url, folder, name, token_header=None):
    os.makedirs(folder, exist_ok=True)
    dest = f"{folder}/{name}"
    if file_ok(dest):
        print(f"  ✅ {name} ya existe — skip"); return dest
    print(f"  ↓ {name}")
    auth = f'--header="Authorization: Bearer {token_header}"' if token_header else ""
    run(f'aria2c -c -x 16 -s 16 -k 1M --console-log-level=error --summary-interval=0 '
        f'{auth} -d "{folder}" -o "{name}" "{url}"')
    ok = file_ok(dest)
    print(f"  {'✅' if ok else '❌'} {name}  ({os.path.getsize(dest)/1024**3:.2f} GB)" if ok else f"  ❌ {name}")
    return dest if ok else None

def dl_hf(url, folder, name):      return dl(url, folder, name, token_header=HF_TOKEN)
def dl_civitai(url, folder, name):
    sep = "&" if "?" in url else "?"
    return dl(f"{url}{sep}token={CIVITAI_KEY}", folder, name)

def dl_parallel(tasks, workers=4):
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
        fs = {ex.submit(fn, url, folder, name): name for fn, url, folder, name in tasks}
        for f in concurrent.futures.as_completed(fs):
            try: f.result()
            except Exception as e: print(f"  ❌ Error: {e}")

def copy_wf(src, name):
    dst = f"{WORKFLOWS}/{name}"
    if os.path.exists(src): shutil.copy2(src, dst); print(f"  ✅ {name} copiado")
    else: print(f"  ⚠️  {name} no encontrado — cópialo manualmente a:\n      {dst}")

def verify(label, path, is_dir=False):
    ok = os.path.isdir(path) if is_dir else file_ok(path)
    print(f"  {'✅' if ok else '❌'} {label}")

# ══════════════════════════════════════════════════════════════
print("\n[SYS] Herramientas del sistema...")
run("apt-get update -qq && apt-get install -y -qq aria2 git wget curl", check=True)

print("\n[1] ComfyUI...")
clone("https://github.com/comfyanonymous/ComfyUI.git", COMFY)
run(f'"{sys.executable}" -m pip install -r "{COMFY}/requirements.txt" -q')
for d in [f"{MODELS}/diffusion_models", f"{MODELS}/text_encoders", f"{MODELS}/vae",
          f"{MODELS}/loras", f"{MODELS}/loras/z-image", f"{MODELS}/upscale_models",
          f"{MODELS}/model_patches", CUSTOM_NODES, WORKFLOWS]:
    os.makedirs(d, exist_ok=True)

print("\n[2] Custom nodes...")
# ── Ya presentes
clone("https://github.com/rgthree/rgthree-comfy.git",          f"{CUSTOM_NODES}/rgthree-comfy")
clone("https://github.com/Joan2022Laurente/node.git",           f"{CUSTOM_NODES}/azzia-nodes")

# ── Requeridos por el workflow
clone("https://github.com/kijai/ComfyUI-KJNodes.git",             f"{CUSTOM_NODES}/causachuperi")
clone("https://github.com/city96/ComfyUI-GGUF.git",             f"{CUSTOM_NODES}/ComfyUI-GGUF")
clone("https://github.com/cubiq/ComfyUI_essentials.git",        f"{CUSTOM_NODES}/ComfyUI_essentials")
clone("https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git", f"{CUSTOM_NODES}/ComfyUI_UltimateSDUpscale")
clone("https://github.com/chrisgoringe/cg-use-everywhere.git",  f"{CUSTOM_NODES}/cg-use-everywhere")
clone("https://github.com/alexopus/ComfyUI-Image-Saver.git",    f"{CUSTOM_NODES}/ComfyUI-Image-Saver")
# ── [NUEVO] Impact Pack — nodos FaceDetectorForEach, GrowMask, InvertMask
clone("https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",    f"{CUSTOM_NODES}/ComfyUI-Impact-Pack")
# ── [NUEVO] Impact Subpack — modelos de detección facial (face_yolov8m.pt)
clone("https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git", f"{CUSTOM_NODES}/ComfyUI-Impact-Subpack")

print("\n[3] Dependencias Python...")
pip_install("accelerate", "einops", "torchvision", "Pillow")
# ── [NUEVO] Requeridas por Impact Pack
pip_install("ultralytics", "dlib", "segment-anything", "mediapipe")
print("  ✅")

print("\n[4] Modelo de detección facial (Impact Pack)...")
BBOX_DIR = f"{MODELS}/ultralytics/bbox"
os.makedirs(BBOX_DIR, exist_ok=True)
dl_hf(
    "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt",
    BBOX_DIR,
    "face_yolov8m.pt"
)

print("\n[5] Descarga paralela de modelos...")
tasks = [
    # ── Encoders ──────────────────────────────────────────────
    (dl_hf, "https://huggingface.co/mradermacher/Qwen3-4B-heretic-GGUF/resolve/main/Qwen3-4B-heretic.Q6_K.gguf",
             f"{MODELS}/text_encoders", "qwen3-4b-heretic-Q6_K.gguf"),
    (dl_hf, "https://huggingface.co/mradermacher/Qwen3-4B-heretic-GGUF/resolve/main/Qwen3-4B-heretic.f16.gguf",
             f"{MODELS}/text_encoders", "qwen3-4b-heretic_F16.gguf"),
    (dl_hf, f"{HF_Z}/text_encoders/qwen_3_4b.safetensors",
             f"{MODELS}/text_encoders", "qwen_3_4b.safetensors"),
    # ── VAE ───────────────────────────────────────────────────
    (dl_hf, f"{HF_Z}/vae/ae.safetensors",
             f"{MODELS}/vae", "ae.safetensors"),
    # ── Upscale model ─────────────────────────────────────────
        (dl_hf, "https://huggingface.co/Phips/4xFaceUpSharpDAT/resolve/main/4xFaceUpSharpDAT.safetensors",
             f"{MODELS}/upscale_models", "4xFaceUpSharpDAT.safetensors"),
    (dl_hf, "https://huggingface.co/Phips/4xFFHQLDAT/resolve/main/4xFFHQLDAT.safetensors",
             f"{MODELS}/upscale_models", "4xFFHQDAT.pth"),
]

# ── Checkpoints Civitai ───────────────────────────────────────
for url, name in CHECKPOINTS:
    if url.strip(): tasks.append((dl_civitai, url.strip(), f"{MODELS}/diffusion_models", name.strip()))
    else:           print(f"  ⚪ {name or 'Checkpoint vacío'}: skip")

# ── LoRAs HF ─────────────────────────────────────────────────
if LORA_HF_ENABLED:
    tasks.append((dl_hf, f"{HF_LORA_BASE}/YummyHDzit.safetensors",
                  f"{MODELS}/loras", "YummyHDzit.safetensors"))
    tasks.append((dl_hf, f"{HF_LORA_BASE}/yummy_kitty.safetensors",
                  f"{MODELS}/loras/z-image", "yummy_kitty.safetensors"))
else:
    print("  ⚪ LoRAs HF: desactivadas")

# ── LoRAs Civitai ─────────────────────────────────────────────
for url, name in LORAS_CIVITAI:
    if url.strip(): tasks.append((dl_civitai, url.strip(), f"{MODELS}/loras", name.strip()))
    else:           print(f"  ⚪ {name or 'LoRA vacío'}: skip")

dl_parallel(tasks, workers=4)

print("\n[6] ControlNet Union...")
CN_NAME = "Z-Image-Turbo-Fun-Controlnet-Union-2.1.safetensors"
CN_URL  = f"https://huggingface.co/alibaba-pai/Z-Image-Turbo-Fun-Controlnet-Union-2.1/resolve/main/{CN_NAME}"
if CONTROLNET_ENABLED:
    if CONTROLNET_SKIP_COMPAT:
        print("  ⚡ Verificación de compatibilidad omitida")
    elif os.path.isdir(f"{MODELS}/model_patches"):
        print("  ✅ Compatibilidad OK")
    else:
        print("  ⚠️  model_patches/ no encontrada — activa CONTROLNET_SKIP_COMPAT=True para forzar")
    dl_hf(CN_URL, f"{MODELS}/model_patches", CN_NAME)
    copy_wf(f"{CUSTOM_NODES}/azzia-nodes/gsl_creator_2.json", "gsl_creator_2.json")
else:
    print("  ⚪ Desactivado")

print("\n[7] Workflows T2I...")
for wf in ["ZIMAGET2I_lora.json", "ZIMAGET2I_ImageHiresFix.json", "ZIMAGET2I_lora_hirexFix.json",
           "ZIMAGET2I_ImageHiresFix_FaceMask.json"]:  # [NUEVO] workflow con máscara facial
    copy_wf(f"{CUSTOM_NODES}/azzia-nodes/{wf}", wf)

# ══════════════════════════════════════════════════════════════
print("\n" + "═"*50 + "\n  VERIFICACIÓN FINAL\n" + "═"*50)

print("\n  — Fijos —")
verify("Qwen3-4B-heretic Q6_K GGUF",  f"{MODELS}/text_encoders/qwen3-4b-heretic-Q6_K.gguf")
verify("Qwen3-4B-heretic F16 GGUF",   f"{MODELS}/text_encoders/qwen3-4b-heretic_F16.gguf")
verify("Qwen3-4B BF16 safetensors",   f"{MODELS}/text_encoders/qwen_3_4b.safetensors")
verify("Flux VAE",                    f"{MODELS}/vae/ae.safetensors")
verify("4xFFHQDAT upscale model",     f"{MODELS}/upscale_models/4xFFHQDAT.pth")

print("\n  — Custom nodes —")
verify("rgthree-comfy",               f"{CUSTOM_NODES}/rgthree-comfy",               is_dir=True)
verify("azzia-nodes",                 f"{CUSTOM_NODES}/azzia-nodes",                 is_dir=True)
verify("ComfyUI-GGUF",                f"{CUSTOM_NODES}/ComfyUI-GGUF",                is_dir=True)
verify("ComfyUI_essentials",          f"{CUSTOM_NODES}/ComfyUI_essentials",          is_dir=True)
verify("ComfyUI_UltimateSDUpscale",   f"{CUSTOM_NODES}/ComfyUI_UltimateSDUpscale",   is_dir=True)
verify("cg-use-everywhere",           f"{CUSTOM_NODES}/cg-use-everywhere",           is_dir=True)
verify("ComfyUI-Image-Saver",         f"{CUSTOM_NODES}/ComfyUI-Image-Saver",         is_dir=True)
verify("ComfyUI-Impact-Pack",         f"{CUSTOM_NODES}/ComfyUI-Impact-Pack",         is_dir=True)   # [NUEVO]
verify("ComfyUI-Impact-Subpack",      f"{CUSTOM_NODES}/ComfyUI-Impact-Subpack",      is_dir=True)   # [NUEVO]

print("\n  — Modelos detección facial —")                                                            # [NUEVO]
verify("face_yolov8m.pt",             f"{MODELS}/ultralytics/bbox/face_yolov8m.pt")                 # [NUEVO]

print("\n  — Checkpoints —")
for url, name in CHECKPOINTS:
    if url.strip(): verify(name, f"{MODELS}/diffusion_models/{name}")
    else:           print(f"  ⚪ {name}: no configurado")

print("\n  — LoRAs —")
if LORA_HF_ENABLED:
    verify("YummyHDzit.safetensors      → loras/",         f"{MODELS}/loras/YummyHDzit.safetensors")
    verify("yummy_kitty.safetensors     → loras/z-image/", f"{MODELS}/loras/z-image/yummy_kitty.safetensors")
else:
    print("  ⚪ LoRAs HF: desactivadas")
for url, name in LORAS_CIVITAI:
    if url.strip(): verify(name, f"{MODELS}/loras/{name}")
    else:           print(f"  ⚪ {name}: no configurado")

print("\n  — ControlNet —")
if CONTROLNET_ENABLED:
    verify(CN_NAME + (" [compat skipped]" if CONTROLNET_SKIP_COMPAT else ""),
           f"{MODELS}/model_patches/{CN_NAME}")
else:
    print("  ⚪ Desactivado")

print("\n✅ SETUP COMPLETO")
print("→ Nota: descarga el UNet z_image_turbo_bf16.safetensors por separado")
print("        y colócalo en:  models/diffusion_models/")
print("→ KSampler : Steps=8-10 · CFG=1.0 · euler · simple")
print("→ CLIP     : qwen3-4b-heretic-Q6_K.gguf  |  VAE: ae.safetensors")

✅ Secrets cargados

[SYS] Herramientas del sistema...


In [ ]:
# 🔒 Lanzar ComfyUI con Tailscale

import os, time
from google.colab import userdata

TAILSCALE_AUTH_KEY = userdata.get('TAILSCALE_KEY')
if not TAILSCALE_AUTH_KEY:
    raise RuntimeError("❌ Falta TAILSCALE_KEY en Secrets de Colab")

# 1. Instalar Tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

# 2. Iniciar daemon
print("🚀 Iniciando tailscaled...")
!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /tmp/tailscaled.log 2>&1 &
time.sleep(5)

# 3. Verificar
print("🔍 Verificando tailscaled...")
!ps aux | grep tailscaled

# 4. Conectar
print("🔗 Conectando a Tailscale...")
!tailscale up --authkey={TAILSCALE_AUTH_KEY} --hostname=colab-comfyui

# 5. Mostrar IP
print("\n" + "="*50)
print("🌐 TU IP PRIVADA DE TAILSCALE ES:")
!tailscale ip -4
print("="*50)

# 6. Lanzar ComfyUI
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --dont-print-server

Installing Tailscale for ubuntu jammy, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ + curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg
tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ + curl -fsSLtee https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list
 /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu jammy
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu jammy main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 https://p

In [ ]:
from IPython.display import display, HTML

display(HTML("""
<script>
const ctx = new (window.AudioContext || window.webkitAudioContext)();
const merger = ctx.createChannelMerger(2);

const gainL = ctx.createGain();
const gainR = ctx.createGain();
gainL.gain.value = 0.4;
gainR.gain.value = 0.4;

const oscL = ctx.createOscillator();
const oscR = ctx.createOscillator();
oscL.type = 'sine';
oscR.type = 'sine';
oscL.frequency.value = 100;   // oído izquierdo
oscR.frequency.value = 133;   // oído derecho (diferencia = 33 Hz Gamma)

oscL.connect(gainL); gainL.connect(merger, 0, 0);
oscR.connect(gainR); gainR.connect(merger, 0, 1);
merger.connect(ctx.destination);

oscL.start();
oscR.start();
</script>
<small style="color:gray">🎧 Binaural activo — 100 Hz / 133 Hz (Gamma 33 Hz)</small>
"""))
# =========================================================
# COMFYUI + PINGGY TUNNEL
# =========================================================

import subprocess
import threading
import time
import re

# =========================================================
# FUNCIÓN TÚNEL PINGGY
# =========================================================

def run_pinggy():
    print("🌐 Iniciando túnel Pinggy...")

    process = subprocess.Popen(
        [
            "ssh",
            "-p", "443",
            "-o", "StrictHostKeyChecking=no",
            "-R0:localhost:8188",
            "a.pinggy.io"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line.strip())

        match = re.search(r"https://[^\s]+\.a\.free\.pinggy\.link", line)

        if match:
            print("\n" + "=" * 70)
            print("🚀 COMFYUI PUBLIC URL:")
            print(match.group(0))
            print("=" * 70 + "\n")

# =========================================================
# LANZAR TÚNEL
# =========================================================

threading.Thread(
    target=run_pinggy,
    daemon=True
).start()

# =========================================================
# ESPERA
# =========================================================

time.sleep(5)

# =========================================================
# INICIAR COMFYUI
# =========================================================

%cd /content/ComfyUI

!python main.py \
    --listen 0.0.0.0 \
    --port 8188 \
    --dont-print-server

